# DrugWatch India — exploring the gold layer

**Purpose (Phase 6 step 1).** Decide what the public dashboards should show, and what they must not
imply. Nothing here is a deliverable; the findings are, and they are written up as aggregates in
`docs/analytics-findings.md`.

**Three rules this notebook follows:**

1. **Gold only** (CLAUDE.md §5). Superset, the risk model and the API all read gold, so the
   exploration that designs the dashboards reads the same tables — otherwise we invent a second
   definition of every number.
2. **Read-only.** `drugwatch_analytics.gold.query()` opens a read-only transaction, so the
   database itself refuses a write from here.
3. **Committed with outputs cleared** (ADR 0003, ADR 0013). The outputs are real CDSCO rows —
   batch numbers and company names — and those do not belong in git. Run the cells to see them.

**Run it:** `cd analytics && uv run --env-file ../.env jupyter lab`

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from drugwatch_analytics.gold import data_freshness, gold_tables, query

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. What is in the warehouse, and how fresh is it

Before any analysis: what are we looking at, and did the pipeline that built it actually succeed?
The freshness answer comes from `ops.pipeline_run` (Phase 5's run log), because that is the only
place that knows whether the last run passed its checks — a timestamp inside gold would only tell
us when rows were written, not whether they were trusted.

In [ ]:
print(gold_tables().to_string(index=False))
print()
freshness = data_freshness()
print(
    freshness[
        ["dag_id", "status", "recorded_at", "silver_rows", "dbt_nodes_passed", "quality_check"]
    ].to_string(index=False)
)

## 2. Findings over time — and the first trap

The obvious chart is "NSQ findings per month". The obvious reading of a rising line is "medicine
quality is getting worse". **That reading is almost certainly wrong**, and this section exists to
work out what else could explain it before a dashboard implies it.

Two confounders to measure, not assume:

- **CDSCO's publishing gaps.** The archive does not cover every calendar month, so a dip may be a
  month nobody published.
- **Testing effort.** More findings can simply mean more laboratories testing more samples. If the
  number of distinct reporting laboratories grows with the findings, the line is about surveillance,
  not quality.

In [ ]:
monthly = query("""
    SELECT alert_month, count(*) AS findings,
           count(*) FILTER (WHERE lab_type = 'Central') AS central,
           count(*) FILTER (WHERE lab_type = 'State')   AS state,
           count(DISTINCT reported_by)                  AS reporting_labs
    FROM gold.fact_nsq_sample
    GROUP BY alert_month ORDER BY alert_month
""")
print(f"{len(monthly)} published months, {monthly['findings'].sum()} findings")
print(monthly.tail(8).to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].bar(monthly["alert_month"], monthly["findings"], color="#4477aa")
axes[0].set_title("NSQ / spurious findings per published alert month")
axes[1].plot(monthly["alert_month"], monthly["reporting_labs"], color="#cc6677")
axes[1].set_title("Distinct reporting laboratories named in that month")
axes[1].set_xticks(monthly["alert_month"][::6])
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()

In [ ]:
# How much of the apparent trend tracks testing effort rather than quality?
yearly = query("""
    SELECT left(alert_month, 4) AS year,
           count(*) AS findings,
           count(DISTINCT reported_by) AS reporting_labs,
           count(DISTINCT alert_month) AS months_published,
           round(count(*)::numeric / count(DISTINCT alert_month), 1) AS findings_per_published_month
    FROM gold.fact_nsq_sample
    GROUP BY 1 ORDER BY 1
""")
print(yearly.to_string(index=False))
print()
print(
    "correlation between findings and reporting labs, by year:",
    round(yearly["findings"].corr(yearly["reporting_labs"]), 3),
)

## 3. Manufacturers — what a "top offenders" chart would really show

Plan §8.2 wanted a `mart_repeat_offenders`; Phase 4 renamed it `mart_repeat_reported_manufacturers`
because "offender" is a verdict and CLAUDE.md §6 forbids one. This section asks the quantitative
version of the same question: **how concentrated is the archive?** If almost every company has a
single finding, then a bar chart of the top 10 tells the public about 10 companies and nothing about
the industry — and it invites exactly the inference the ethics rules prohibit.

In [ ]:
per_maker = query("""
    SELECT m.canonical_name, m.state_name, count(*) AS findings,
           count(DISTINCT f.alert_month) AS alert_months
    FROM gold.fact_nsq_sample f
    JOIN gold.dim_manufacturer m USING (manufacturer_id)
    GROUP BY 1, 2
""")
total = per_maker["findings"].sum()
print(
    f"{len(per_maker)} manufacturers carry {total} findings "
    f"(mean {total / len(per_maker):.1f}, median {per_maker['findings'].median():.0f})"
)
for share in (0.01, 0.05, 0.10):
    top = per_maker.nlargest(max(1, int(len(per_maker) * share)), "findings")
    print(
        f"  top {share:.0%} of companies ({len(top):4}) hold {top['findings'].sum() / total:5.1%} of findings"
    )
print()
counts = per_maker["findings"].value_counts().sort_index()
print("companies by number of findings:")
print(counts.head(6).to_string())
print(
    f"  ... companies with a single finding: {counts.get(1, 0)} "
    f"({counts.get(1, 0) / len(per_maker):.0%} of companies)"
)

In [ ]:
top10 = per_maker.nlargest(10, "alert_months").sort_values("alert_months")
fig, ax = plt.subplots(figsize=(11, 4))
ax.barh(top10["canonical_name"].str.slice(0, 45), top10["alert_months"], color="#4477aa")
ax.set_title("Most frequently reported manufacturers, by number of separate alert months")
ax.set_xlabel("alert months in which the company was reported (NOT a measure of quality)")
plt.tight_layout()

## 4. Failure categories

ADR 0009 mapped free-text lab findings into 15 categories, and 22.6% of findings name more than one
failed test — so a pie chart of categories cannot add to 100%. Whatever the dashboard shows has to
say *mentions*, not *findings*.

In [ ]:
categories = query("""
    SELECT category, count(*) AS mentions,
           count(DISTINCT manufacturer_id) AS manufacturers
    FROM gold.fact_nsq_sample, unnest(failure_categories) AS category
    GROUP BY 1 ORDER BY 2 DESC
""")
multi = query("""
    SELECT round(100.0 * count(*) FILTER (WHERE array_length(failure_categories, 1) > 1) / count(*), 1) AS pct_multi,
           round(avg(array_length(failure_categories, 1)), 2) AS mean_categories
    FROM gold.fact_nsq_sample
""")
print(categories.to_string(index=False))
print()
print(multi.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(categories["category"], categories["mentions"], color="#117733")
ax.set_title(
    "Failure category mentions (a finding can name several, so these do not sum to the row count)"
)
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()

## 5. States — the distinction a dashboard must never blur

Phase 4 found that 498 findings tested in West Bengal were made in Himachal Pradesh, because CDSCO's
central laboratories test samples drawn nationwide. So "reports by state" has two completely
different meanings, and a single map would be read as the wrong one.

In [ ]:
states = query("""
    SELECT state_name, reports_made_here, reports_tested_here,
           state_lab_reports_tested_here, manufacturers_here
    FROM gold.mart_state_summary
    WHERE reports_made_here > 0 OR reports_tested_here > 0
    ORDER BY reports_made_here DESC
""")
print(states.head(12).to_string(index=False))
print()
mismatch = query("""
    SELECT count(*) AS findings_whose_lab_state_differs_from_maker_state
    FROM gold.fact_nsq_sample f
    JOIN gold.dim_manufacturer m USING (manufacturer_id)
    WHERE f.reported_state_name IS NOT NULL AND m.state_name IS NOT NULL
      AND f.reported_state_name <> m.state_name
""")
print(mismatch.to_string(index=False))

top = states.head(12).sort_values("reports_made_here")
fig, ax = plt.subplots(figsize=(11, 5))
y = range(len(top))
ax.barh(
    [i - 0.2 for i in y], top["reports_made_here"], height=0.4, label="made here", color="#4477aa"
)
ax.barh(
    [i + 0.2 for i in y],
    top["reports_tested_here"],
    height=0.4,
    label="tested here",
    color="#cc6677",
)
ax.set_yticks(list(y))
ax.set_yticklabels(top["state_name"])
ax.set_title("Two different questions, never to be summed")
ax.legend()
plt.tight_layout()

## 6. The medicines, and the spurious subset

Spurious (counterfeit) findings name the company printed on the fake pack, not whoever made it, so
Phase 3 and Phase 4 keep them unattributed (CLAUDE.md §6). They are a small but important slice, and
they need their own treatment on any dashboard.

In [ ]:
forms = query("""
    SELECT coalesce(d.dosage_form, '(not stated)') AS dosage_form, count(*) AS findings
    FROM gold.fact_nsq_sample f JOIN gold.dim_drug d USING (drug_id)
    GROUP BY 1 ORDER BY 2 DESC LIMIT 12
""")
spurious = query("""
    SELECT count(*) AS spurious_findings,
           count(*) FILTER (WHERE manufacturer_id IS NOT NULL) AS wrongly_attributed,
           count(*) FILTER (WHERE name_on_pack IS NOT NULL) AS with_a_name_on_pack,
           min(alert_month) AS first_month, max(alert_month) AS last_month
    FROM gold.fact_nsq_sample WHERE is_spurious
""")
print(forms.to_string(index=False))
print()
print(spurious.to_string(index=False))

## 7. The gaps — what the dashboards must disclose

A public dashboard that does not state its gaps invites the reader to treat absence as safety.
"Not found" never means "guaranteed safe" (CLAUDE.md §6).

In [ ]:
gaps = query("""
    SELECT
      (SELECT count(*) FROM gold.fact_nsq_sample) AS findings,
      (SELECT count(DISTINCT alert_month) FROM gold.fact_nsq_sample) AS months_published,
      (SELECT (extract(year from max(alert_month_start)) * 12 + extract(month from max(alert_month_start))
             - extract(year from min(alert_month_start)) * 12 - extract(month from min(alert_month_start)) + 1)::int
         FROM gold.fact_nsq_sample) AS calendar_months_spanned,
      (SELECT count(*) FROM gold.fact_nsq_sample WHERE manufacturer_id IS NULL) AS findings_with_no_manufacturer,
      (SELECT count(*) FROM gold.dim_manufacturer WHERE state_name IS NULL) AS manufacturers_with_no_state,
      (SELECT count(*) FROM gold.fact_nsq_sample WHERE mfg_date IS NULL OR expiry_date IS NULL) AS findings_missing_a_date,
      (SELECT count(*) FROM gold.fact_nsq_sample WHERE source_dataset IN ('central-nsq','state-nsq','combined')) AS pdf_era_findings
""").T
gaps.columns = ["value"]
print(gaps.to_string())
print()
by_source = query("""
    SELECT source_dataset, extraction_method, count(*) AS findings,
           min(alert_month) AS first_month, max(alert_month) AS last_month
    FROM gold.fact_nsq_sample GROUP BY 1, 2 ORDER BY 3 DESC
""")
print(by_source.to_string(index=False))

## 8. What this means for the dashboards

The conclusions drawn from the cells above are written up as aggregates in
**`docs/analytics-findings.md`** — that file is the durable output of this notebook, because this
one is committed with its outputs cleared.

The four that change what gets built:

1. **The rising line is testing effort.** Findings and the number of reporting laboratories
   correlate at r = 0.964 by year. A bare findings-per-month chart would be read as "medicine is
   getting worse", which this data does not support.
2. **The archive is a long tail.** 58% of companies appear exactly once; the median company has one
   finding. A top-10 chart describes 10 of 2,609 companies.
3. **The two state questions are almost never the same.** Where both are known they differ in 92.7%
   of findings. Two charts, never one map, never summed.
4. **Categories are mentions, not shares.** 21.8% of findings name more than one failed test, so
   they cannot be a pie chart. `Spurious: 77 mentions, 0 manufacturers` is the ethics rule visible
   in the data.

Re-run this notebook whenever the archive grows; if any of those four stops being true, the
dashboards built on them need revisiting.